# 12 · Capstone project: from messy export to business report

The shop's finance team sends you `orders_messy.csv` — a raw export with problems — and wants a report.
Build a clean pipeline step by step. Every step has a check. Use anything from the course.

**Known problems in the file:**
* `status` has inconsistent case and stray spaces (`" Completed "`, `"COMPLETED"`, `"completed"`)
* `quantity` sometimes contains `N/A` or a negative number
* some rows have an empty `order_ts`
* some rows are exact duplicates

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA}/products.csv", header=True, inferSchema=True)

### Exercise 1
Read `orders_messy.csv` into `raw` with a header and **all columns as strings** (no schema inference). Explore it!

In [ ]:
raw = None  # TODO

In [ ]:
# Check — run me after your answer
assert raw.count() == 1050
assert all(t == "string" for _, t in raw.dtypes)
print("Correct!")

### Exercise 2
Create `clean` with columns **`order_id, customer_id, product_id, quantity, order_ts, status`** where:
* the three id columns and `quantity` are **int**, `order_ts` is a **timestamp** (use `try_cast` so bad values become null)
* `status` is trimmed and lower-cased
* rows with null/≤0 quantity or null `order_ts` are removed
* exact duplicate rows are removed

In [ ]:
clean = None  # TODO

In [ ]:
# Check — run me after your answer
assert clean.columns == ["order_id", "customer_id", "product_id", "quantity", "order_ts", "status"]
assert dict(clean.dtypes) == {"order_id": "int", "customer_id": "int", "product_id": "int",
                              "quantity": "int", "order_ts": "timestamp", "status": "string"}
assert sorted(r.status for r in clean.select("status").distinct().collect()) == ["cancelled", "completed", "returned"]
assert clean.count() == 943, clean.count()
print("Correct!")

### Exercise 3
Create `sales`: **completed** orders from `clean`, joined to products and customers, with a **`revenue`** column (quantity × price) and null countries replaced by **`"Unknown"`**.

In [ ]:
sales = None  # TODO

In [ ]:
# Check — run me after your answer
assert sales.filter(F.col("country").isNull()).count() == 0
assert abs(sales.agg(F.sum("revenue")).first()[0] - 266509.41) < 0.05
print("Correct!")

### Exercise 4
Create `monthly_country`: revenue per **month** (string like `"2024-03"`) and **country**. Columns **`month`, `country`, `revenue`** (rounded 2 dp), sorted by month then country.

In [ ]:
monthly_country = None  # TODO

In [ ]:
# Check — run me after your answer
assert monthly_country.columns == ["month", "country", "revenue"]
assert monthly_country.count() == 96
m, c, r = ['2024-01', 'Brazil', 1812.04]
first = monthly_country.first()
assert (first.month, first.country) == (m, c) and abs(first.revenue - r) < 0.05, first
print("Correct!")

### Exercise 5
Create `top_customers`: the **5** customers with the highest total revenue. Columns **`customer_id`, `name`, `revenue`, `n_orders`, `rank`** (rank 1 = highest), ordered by rank.

In [ ]:
top_customers = None  # TODO

In [ ]:
# Check — run me after your answer
assert top_customers.columns == ["customer_id", "name", "revenue", "n_orders", "rank"]
assert [r.customer_id for r in top_customers.collect()] == [163, 11, 19, 122, 15]
print("Correct!")

### Exercise 6
Store in `repeat_rate` (a float rounded to 3 dp) the share of customers in `sales` who made **2 or more** completed orders.

In [ ]:
repeat_rate = None  # TODO

In [ ]:
# Check — run me after your answer
assert repeat_rate == 0.927, repeat_rate
print("Correct!")

### Exercise 7
Save `monthly_country` as Parquet to `f"{OUT}/report/monthly_country"` (overwrite), **partitioned by `country`**.

In [ ]:
# TODO

In [ ]:
# Check — run me after your answer
back = spark.read.parquet(f"{OUT}/report/monthly_country")
assert back.count() == 96
assert any(d.startswith("country=") for d in os.listdir(f"{OUT}/report/monthly_country"))
print("Correct!")

## You've finished the core course!

### Where to go next
* **Databricks Free Edition** — run Spark in the cloud on real clusters, for free.
* **Delta Lake** — ACID tables on top of Parquet (`MERGE`, time travel); the standard on Databricks.
* **Structured Streaming** — lesson 13 in this folder gives you a taste.
* **MLlib** (`pyspark.ml`) — distributed machine-learning pipelines.
* **Spark Connect** — the thin-client architecture behind modern Spark deployments.
* Official docs: <https://spark.apache.org/docs/latest/api/python/>
* Certification: *Databricks Certified Associate Developer for Apache Spark* covers exactly this material.

---
Stuck? Worked answers (with outputs) are in [`../solutions/12_capstone_project.ipynb`](../solutions/12_capstone_project.ipynb).